# A06 · CNNs & Sequence Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adib2024/aws-architect-path/blob/main/notebooks/a06_cnn_rnn.ipynb)

Before transformers, two architectures ruled: CNNs for images and RNNs for sequences. You'll compute convolution shapes, parameters and FLOPs, train a CNN, and see exactly why RNNs struggle with long sequences, which is the problem attention solves.

Part of the **AI Engineer Path** (https://adib2024.github.io/aws-architect-path/#a06). In Colab, use **Runtime → Change runtime type → T4 GPU** for anything that trains.

**You will:**
- Compute conv output size, parameter count and FLOPs from the hyperparameters.
- Train a small CNN to about 90% on Fashion-MNIST.
- Explain the receptive field and why depth grows it.
- Measure vanishing gradients in an RNN and explain how LSTMs and attention mitigate them.

## The math

### Convolution output size

$$H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1$$

ResNet's first layer: $224\times224\times3$ input, $K=7,\ S=2,\ P=3,\ C_{out}=64$.

**Worked example**

1. $H_{out} = \lfloor (224 + 6 - 7)/2 \rfloor + 1 = 111 + 1 = 112$
2. params $= K^2 C_{in} C_{out} = 7\cdot7\cdot3\cdot64 = 9{,}408$ (no bias (BatchNorm follows))
3. MACs $= H_{out}W_{out}\,K^2C_{in}C_{out} = 112\cdot112\cdot147\cdot64 \approx 118\text{M}$ ($\approx 0.24$ GFLOPs for one layer)

### Why RNN gradients vanish

$$\frac{\partial h_T}{\partial h_t} = \prod_{k=t+1}^{T} \mathrm{diag}\!\left(1 - h_k^2\right) W_h$$

A product of $T - t$ Jacobians. If their norm is below 1, the gradient decays exponentially with distance; above 1, it explodes.

**Worked example**

1. $\|J\| \approx 0.9,\ T - t = 50 \Rightarrow 0.9^{50} \approx 0.005$ (almost no learning signal from 50 tokens back)
2. attention: any token reaches any other in one step, path length $O(1)$ (the core reason transformers won (A08))

In [ ]:
%pip install -q torchvision

## Lab: Train a CNN and watch an RNN forget

Confirm the conv formulas in code, train to ~90% test accuracy, then measure gradient decay through time.

### Step 1

Check the worked example: ResNet's first conv.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
conv = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
x = torch.randn(1, 3, 224, 224)
print(conv(x).shape)                                  # torch.Size([1, 64, 112, 112])
print(sum(p.numel() for p in conv.parameters()))      # 9408

### Step 2

Train the CNN from the diagram on Fashion-MNIST for two epochs.

In [ ]:
from torchvision import datasets, transforms
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tf = transforms.ToTensor()
train = datasets.FashionMNIST('data', train=True, download=True, transform=tf)
test = datasets.FashionMNIST('data', train=False, download=True, transform=tf)
train_dl = torch.utils.data.DataLoader(train, batch_size=128, shuffle=True)
test_dl = torch.utils.data.DataLoader(test, batch_size=512)
model = nn.Sequential(
    nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 32x14x14
    nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 64x7x7
    nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for epoch in range(2):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); correct = 0
    with torch.no_grad():
        for xb, yb in test_dl:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
    print(f"epoch {epoch}  last loss {loss.item():.3f}  test acc {correct / len(test):.2%}")

### Step 3

Measure how much gradient reaches each input step of a 100-step RNN, then compare an LSTM.

In [ ]:
torch.manual_seed(0)
for cell in [nn.RNN, nn.LSTM]:
    rnn = cell(input_size=8, hidden_size=32, batch_first=True)
    x = torch.randn(1, 100, 8, requires_grad=True)
    out, _ = rnn(x)
    out[0, -1].sum().backward()          # loss depends on the last output only
    g = x.grad[0].norm(dim=1)
    print(f"{cell.__name__:5s} grad at t=99: {g[99]:.2e}   t=50: {g[50]:.2e}   t=0: {g[0]:.2e}")

### Clean up

In Colab, disconnect the runtime when finished.

## Check yourself

**1. Input $32\times32$, $K = 5$, $S = 1$, $P = 0$. Output size?**

A. 32  
B. 28  
C. 27  
D. 30

<details><summary>Answer</summary>

**B.** $(32 - 5)/1 + 1 = 28$.

</details>

**2. Parameters of Conv2d(64, 128, kernel_size=3) with bias?**

A. 73,728  
B. 73,856  
C. 1,152  
D. 8,192

<details><summary>Answer</summary>

**B.** $3\cdot3\cdot64\cdot128 + 128 = 73{,}728 + 128 = 73{,}856$.

</details>

**3. Main reason RNNs were replaced by transformers for language?**

A. RNNs can't handle text  
B. Sequential computation (no parallelism over time) and long gradient paths  
C. RNNs have too many parameters  
D. RNNs need labelled data

<details><summary>Answer</summary>

**B.** Attention connects any two positions in one step and computes all positions in parallel.

</details>